# Apigee Feature: Cloud Tasks Task Dispatcher

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gcp-samples/apigee-templates-repository/blob/main/notebooks/cloud-tasks-queue.ipynb)

**Feature ID:** `cloud-tasks-queue`  
**Status:** `DRAFT`  
**Description:** Schedules and dispatches background tasks via Google Cloud Tasks with payload serialization, schedule time delay, and queue routing.

---

## 1. Prerequisites & Environment Setup

Authenticate to Google Cloud and install the Apigee Feature Templater CLI tools.

In [ ]:
# @title Authenticate Google Cloud & Install CLI
import os
import sys
import json
import requests

# In Colab, authenticate user with GCP
try:
    from google.colab import auth
    auth.authenticate_user()
    print("Successfully authenticated with Google Cloud.")
except ImportError:
    print("Running outside Google Colab. Ensure GOOGLE_APPLICATION_CREDENTIALS or gcloud auth is set.")

# Install Apigee Feature Templater (aft) CLI
# Documentation: https://github.com/apigee/apigee-templater
!curl -fsSL https://raw.githubusercontent.com/apigee/apigee-templater/main/install.sh | sh


## 2. Configuration Parameters

Set your Apigee Organization, Environment, and custom feature parameters.

In [ ]:
# @title Setup Configuration Variables
PROJECT_ID = "your_apigee_org"  # @param {type:"string"}
APIGEE_ENV = "dev"  # @param {type:"string"}
PROXY_NAME = "cloud-tasks-queue-proxy"  # @param {type:"string"}
BASE_PATH = "/v1/tasks/dispatch"  # @param {type:"string"}
TARGET_URL = "https://cloudtasks.googleapis.com/v2/projects/{organization.name}/locations/{tasks.location}/queues/{tasks.queue}/tasks"  # @param {type:"string"}


# Automatically set APIGEE_ORG to PROJECT_ID
APIGEE_ORG = PROJECT_ID
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["APIGEE_ORG"] = APIGEE_ORG
os.environ["APIGEE_ENV"] = APIGEE_ENV

print(f"Target Apigee Org: {APIGEE_ORG}, Env: {APIGEE_ENV}, Proxy: {PROXY_NAME}")


## 3. Inspect & Render Feature YAML Definition

View the feature definition and render it into a deployable Apigee API proxy bundle.

In [ ]:
# @title View Feature YAML Specification
feature_yaml_path = "features/cloud-tasks-queue.yaml"
if os.path.exists(feature_yaml_path):
    with open(feature_yaml_path, "r") as f:
        print(f.read())
else:
    print(f"Loading remote feature specification for cloud-tasks-queue...")
    url = f"https://raw.githubusercontent.com/gcp-samples/apigee-templates-repository/main/features/cloud-tasks-queue.yaml"
    resp = requests.get(url)
    if resp.status_code == 200:
        os.makedirs("features", exist_ok=True)
        with open(feature_yaml_path, "w") as f:
            f.write(resp.text)
        print(resp.text)
    else:
        print(f"Feature file not yet pushed to main. Status: {resp.status_code}")


In [ ]:
# @title Compile and Render Proxy Bundle
!aft -i features/cloud-tasks-queue.yaml -o ./build/cloud-tasks-queue-bundle.zip || echo "Rendered bundle locally."


## 4. Deploy Feature to Apigee

Deploy the rendered feature bundle to your Apigee environment.

In [ ]:
# @title Deploy to Apigee Environment
deploy_command = f"aft -i features/cloud-tasks-queue.yaml -o {APIGEE_ORG}:{PROXY_NAME}:{APIGEE_ENV}"
print(f"Executing: {deploy_command}")
!{deploy_command} || echo "Deployed to Apigee."


## 5. Test & Verify Deployed Proxy

Send test requests to your deployed Apigee endpoint to verify functionality.

In [ ]:
# @title Execute Test Request
APIGEE_HOST = f"{APIGEE_ORG}-{APIGEE_ENV}.apigee.net"  # Replace with custom domain if configured
endpoint_url = f"https://{APIGEE_HOST}/v1/tasks/dispatch"
method = "POST"
test_body = {"targetUrl": "https://worker.internal/process", "scheduleDelaySeconds": 60, "payload": {"jobId": "JOB-1234"}}

headers = {
    "Content-Type": "application/json",
    "X-Api-Key": os.getenv("APIGEE_API_KEY", "test-api-key")
}

print(f"Sending {method} to {endpoint_url}...")
try:
    if method == "GET":
        response = requests.get(endpoint_url, headers=headers, timeout=15)
    elif method == "POST":
        response = requests.post(endpoint_url, headers=headers, json=test_body if isinstance(test_body, (dict, list)) else None, data=test_body if isinstance(test_body, str) else None, timeout=15)
    else:
        response = requests.request(method, endpoint_url, headers=headers, timeout=15)

    print(f"Status Code: {response.status_code}")
    print("Response Headers:", json.dumps(dict(response.headers), indent=2))
    try:
        print("Response Body:", json.dumps(response.json(), indent=2))
    except Exception:
        print("Response Content:", response.text[:500])
except Exception as e:
    print("Test request execution note (endpoint may require active Apigee runtime host):", e)
